# Demo 1: Apache Airflow – az első DAG

**Kapcsolódó diák:** 12–23 (Apache Airflow architektúra, operátorok, TaskFlow API)

**Előfeltétel:** `docker compose up -d` – Airflow UI: http://localhost:8080 (admin/admin)

Ez a notebook lépésről lépésre vezet végig az Airflow legfontosabb fogalmain:
1. Orchestration fogalma – cron vs DAG
2. DAG struktúra és topológiai sorrend
3. Airflow architektúra elemei
4. DagBag – Scheduler betöltési mechanizmus
5. `default_args` – minden task alapértelmezései
6. Klasszikus `PythonOperator` és `>>` dependency chain
7. TaskFlow API – `@task` dekorátor
8. XCom – task-ok közötti adatátadás
9. Retry és hibakezelés
10. `execution_date` vs valós futási idő

**Csomagtelepítés:**

In [2]:
!pip install -q apache-airflow==2.9.1 duckdb --constraint \
  https://raw.githubusercontent.com/apache/airflow/constraints-2.9.1/constraints-3.11.txt

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pydantic 2.12.5 requires typing-extensions>=4.14.1, but you have typing-extensions 4.11.0 which is incompatible.
prefect 2.19.0 requires anyio<4.0.0,>=3.7.1, but you have anyio 4.3.0 which is incompatible.
prefect 2.19.0 requires httpcore<2.0.0,>=1.0.5, but you have httpcore 0.16.3 which is incompatible.
prefect 2.19.0 requires importlib-resources<6.2.0,>=6.1.3, but you have importlib-resources 6.4.0 which is incompatible.
prefect 2.19.0 requires pendulum<3.0; python_version < "3.12", but you have pendulum 3.0.0 which is incompatible.
prefect 2.19.0 requires uvicorn<0.29.0,>=0.14.0, but you have uvicorn 0.42.0 which is incompatible.
typing-inspection 0.4.2 requires typing-extensions>=4.12.0, but you have typing-extensions 4.11.0 which is incompatible.
pydantic-core 2.41.5 requires typing-extensions>=4.14.1, but yo

## 1. Mi az az Orchestration? Cron job vs DAG

**Orchestration** = az a folyamat, amely meghatározza, *mikor* és *milyen sorrendben* futnak le az adatfeldolgozó lépések.

### Cron job
- Időzített parancs futtatása (pl. `0 6 * * *` = minden reggel 6-kor)
- Nem tud a többi job-ról – nincsenek függőségek
- Ha egy job megbukik, a következő mégis elindul
- Nincs beépített retry, nincs UI, nincs naplózás

### DAG (Directed Acyclic Graph)
- **Irányított**: a task-ok között irány van (A → B: B függ A-tól)
- **Aciklikus**: nincs kör (nem lehet A → B → A)
- **Gráf**: több elágazás és összevonás is lehetséges
- Ha egy task megbukik → a tőle függő task-ok *nem* indulnak el

**Analógia:** a cron job olyan, mint egy ébresztőóra – minden reggel megszólal, nem érdekli,
hogy az előző nap elvégezted-e a dolgod. A DAG olyan, mint egy recept:
a tészta csak akkor kerülhet a sütőbe, ha az összes hozzávalót előkészítetted.

In [3]:
# Cron analógia – mi történik, ha az egyik lépés megbukik?
import time

def run_cron_jobs():
    """Cron-szerű futtatás: minden job elindul, függetlenül az előzőtől."""
    jobs = [
        ('extract_data',   lambda: (_ for _ in ()).throw(RuntimeError('DB kapcsolat meghiúsult'))),
        ('transform_data', lambda: print('  transform_data: fut (nem tudja, hogy extract megbukott!)')),
        ('load_data',      lambda: print('  load_data:      fut (üres adatot tölt be!)')),
    ]
    print('=== CRON futtatás ===')
    for name, job in jobs:
        try:
            job()
            print(f'  {name}: OK')
        except Exception as e:
            print(f'  {name}: HIBA – {e}  (de a következő job MÉGIS elindul!)')

def run_dag_jobs():
    """DAG-szerű futtatás: ha egy task megbukik, a downstream task-ok leállnak."""
    tasks = {
        'extract_data':   {'fn': lambda: (_ for _ in ()).throw(RuntimeError('DB kapcsolat meghiúsult')), 'deps': []},
        'transform_data': {'fn': lambda: print('  transform_data: fut'), 'deps': ['extract_data']},
        'load_data':      {'fn': lambda: print('  load_data: fut'),      'deps': ['transform_data']},
    }
    failed = set()
    print('\n=== DAG futtatás ===')
    for name, cfg in tasks.items():
        if any(d in failed for d in cfg['deps']):
            print(f'  {name}: KIHAGYVA (upstream task megbukott)')
            failed.add(name)
            continue
        try:
            cfg['fn']()
            print(f'  {name}: OK')
        except Exception as e:
            print(f'  {name}: HIBA – {e}')
            failed.add(name)

run_cron_jobs()
run_dag_jobs()

=== CRON futtatás ===
  extract_data: HIBA – DB kapcsolat meghiúsult  (de a következő job MÉGIS elindul!)
  transform_data: fut (nem tudja, hogy extract megbukott!)
  transform_data: OK
  load_data:      fut (üres adatot tölt be!)
  load_data: OK

=== DAG futtatás ===
  extract_data: HIBA – DB kapcsolat meghiúsult
  transform_data: KIHAGYVA (upstream task megbukott)
  load_data: KIHAGYVA (upstream task megbukott)


### Mi történt?

- **Cron esetén:** az `extract_data` megbukott, de a `transform_data` és `load_data` mégis lefutott –
  üres/hibás adatot dolgoztak fel, silently.
- **DAG esetén:** az `extract_data` hibája leállította a teljes pipeline-t.
  A downstream task-ok `KIHAGYVA` (upstream_failed) státuszba kerültek.

Ez az egyik legfontosabb érv az orchestrator mellett: **a hibák nem terjednek tovább.**

## 2. Mi az a DAG? Fogalom és hétköznapi analógia

A **DAG** (Directed Acyclic Graph) egy matematikai struktúra:
- **Csúcsok (nodes):** a task-ok
- **Élek (edges):** a függőségek (melyik task után jön a másik)
- **Irányított:** minden él egyik irányba mutat
- **Aciklikus:** nem lehet visszafelé jutni egy csúcshoz

**Recept analógia:**
```
lisztet_bemér  tojást_feltör
      \             /
       tésztát_gyúr
            |
       sütőbe_tesz
            |
         tálal
```
A `lisztet_bemér` és `tojást_feltör` párhuzamosan futhat (nincs közöttük függőség).
A `tésztát_gyúr` csak akkor kezdhet el, ha mindkettő kész.

**Miért nem lehet kör?**  
Ha A → B → C → A lenne, sosem tudnánk eldönteni, melyiket indítsuk el elsőnek.
Az Airflow `CycleError`-t dob, ha körkörös függőséget észlel.

In [4]:
# DAG struktúra szimulálása – dict-alapú gráf + topológiai sorrend (Kahn-algoritmus)
# (NetworkX nélkül, hogy lássuk a mögöttes logikát)

from collections import deque

def topological_sort(graph):
    """Kahn-algoritmus: BFS-alapú topológiai sorrend.
    graph: {task_id: [függőségeinek listája]}
    """
    # In-degree számolás: hány task mutat egy csúcsra?
    in_degree = {node: 0 for node in graph}
    for node, deps in graph.items():
        for dep in deps:
            in_degree[node] += 1  # node-nak van dep-függősége

    # Nullás in-degree-ű csúcsokkal kezdünk (nincsenek függőségeik)
    queue = deque([n for n, d in in_degree.items() if d == 0])
    order = []

    while queue:
        node = queue.popleft()
        order.append(node)
        # Minden task-nál csökkentjük azok in-degree-jét, akik rá vártak
        for candidate, deps in graph.items():
            if node in deps:
                in_degree[candidate] -= 1
                if in_degree[candidate] == 0:
                    queue.append(candidate)

    if len(order) != len(graph):
        raise ValueError('Kör van a gráfban! (CycleError)')
    return order

# ETL pipeline gráfja: {task: [amitől függ]}
etl_graph = {
    'extract':   [],
    'validate':  ['extract'],
    'transform': ['validate'],
    'load_dw':   ['transform'],
    'load_mart': ['transform'],   # párhuzamos ág!
    'notify':    ['load_dw', 'load_mart'],
}

order = topological_sort(etl_graph)
print('Topológiai futási sorrend:')
for i, task in enumerate(order, 1):
    deps = etl_graph[task]
    dep_str = f'  ← {deps}' if deps else '  (kezdőpont)'
    print(f'  {i}. {task}{dep_str}')

# Körkörös gráf tesztelése
print('\nKörkörös gráf tesztelése...')
try:
    topological_sort({'A': ['C'], 'B': ['A'], 'C': ['B']})
except ValueError as e:
    print(f'  Hiba: {e}')

Topológiai futási sorrend:
  1. extract  (kezdőpont)
  2. validate  ← ['extract']
  3. transform  ← ['validate']
  4. load_dw  ← ['transform']
  5. load_mart  ← ['transform']
  6. notify  ← ['load_dw', 'load_mart']

Körkörös gráf tesztelése...
  Hiba: Kör van a gráfban! (CycleError)


### Miért fontos a ciklus-mentesség?

Az Airflow a DAG-ot betöltéskor ellenőrzi: ha kört talál, `AirflowDagCycleException`-t dob
és a DAG **nem jelenik meg az UI-on**.

A topológiai sorrend garantálja, hogy mindig pontosan tudjuk,
melyik task-ot lehet következőnek indítani (az összes függősége már futott).
Ez az alap, amire az Airflow Scheduler épül.

## 3. Airflow architektúra elemei

```
Docker Compose
├── postgres        – Metadata DB: DAG runs, task states, XCom értékek, kapcsolatok
├── airflow-init    – Egyszeri DB migráció (alembic) + admin user létrehozása
├── webserver       – Flask-alapú UI (http://localhost:8080), REST API
└── scheduler       – DAG fájlok figyelése (DagBag), Run-ok ütemezése, Executor hívás
```

| Komponens    | Feladata                                              | Kommunikáció           |
|:-------------|:------------------------------------------------------|:-----------------------|
| Scheduler    | DAG-ok betöltése, futtatás ütemezése                  | Metadata DB írás       |
| Webserver    | UI megjelenítés, manuális trigger                     | Metadata DB olvasás    |
| Executor     | Task-ok tényleges elindítása (subprocess / pod / …)   | Scheduler hívja        |
| Metadata DB  | Állapot tárolás (PostgreSQL / SQLite)                 | Mindenki ír/olvas      |

**LocalExecutor:** a Scheduler közvetlenül indít subprocess-eket a task-okhoz – nincs külön Worker.
Kis-közepes terhelésre elegendő; nagy skálán `CeleryExecutor` (worker pool)
vagy `KubernetesExecutor` (1 pod / task) ajánlott.

In [5]:
import importlib.util, sys

# DAG fájl betöltése – a Scheduler pontosan így csinálja (importlib)
# Ez azt jelenti: a DAG fájlnak importálhatónak kell lennie mellékhatás nélkül!
spec = importlib.util.spec_from_file_location('etl_pipeline', 'dags/etl_pipeline.py')
mod  = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

print('DAG fájl sikeresen importálva.')
print('DAG objektum:', mod.dag)
print('Schedule:', mod.dag.schedule_interval)
print('Catchup:', mod.dag.catchup)

DAG fájl sikeresen importálva.
DAG objektum: <DAG: etl_pipeline>
Schedule: @daily
Catchup: False


## 4. DagBag – a Scheduler betöltési mechanizmusa

`DagBag` az az osztály, amely a Scheduler által is használt módon tölt be DAG fájlokat egy mappából.

**Hogyan működik?**
1. Végigmegy a `dag_folder` minden `.py` fájlján
2. Minden fájlt `importlib`-bel betölt
3. Megkeresi a `DAG(...)` objektumokat a modul namespace-ében
4. Ha import hiba van → rögzíti `import_errors`-ban (nem dobja el a többit!)

**CI/CD relevancia:** a `DAG import check` ezt a mechanizmust futtatja PR-onként.
Ha `import_errors` nem üres → a pipeline megbukik.

In [6]:
import sys
sys.path.insert(0, '.')

from airflow.models import DagBag

bag = DagBag(dag_folder='dags', include_examples=False)

print(f'Import hibák: {bag.import_errors or "Nincs"}')
print(f'Betöltött DAG-ok: {list(bag.dags.keys())}')

for dag_id, dag in bag.dags.items():
    tasks = [t.task_id for t in dag.topological_sort()]
    print(f'\n{dag_id}:')
    print(f'  Schedule: {dag.schedule_interval}')
    print(f'  Tasks:    {" → ".join(tasks)}')
    print(f'  Catchup:  {dag.catchup}')

[2026-03-31T07:36:31.176+0000] {dagbag.py:545} INFO - Filling up the DagBag from dags
[2026-03-31T07:36:31.524+0000] {dagbag.py:514} ERROR - Exception bagging dag: etl_pipeline
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/site-packages/airflow/models/dagbag.py", line 505, in _bag_dag
    raise AirflowDagDuplicatedIdException(
airflow.exceptions.AirflowDagDuplicatedIdException: Ignoring DAG etl_pipeline from dags/.ipynb_checkpoints/etl_pipeline-checkpoint.py - also found in dags/etl_pipeline.py
[2026-03-31T07:36:31.525+0000] {dagbag.py:452} ERROR - Failed to bag_dag: dags/.ipynb_checkpoints/etl_pipeline-checkpoint.py
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/site-packages/airflow/models/dagbag.py", line 448, in _process_modules
    self.bag_dag(dag=dag, root_dag=dag)
  File "/opt/conda/lib/python3.11/site-packages/airflow/models/dagbag.py", line 467, in bag_dag
    self._bag_dag(dag=dag, root_dag=root_dag, recursive=True)
  File "/opt/c

### Mi történt?

- A `DagBag` betöltötte mindkét DAG-ot a `dags/` mappából.
- Az `import_errors` üres → a DAG-ok szintaktikailag és szemantikailag helyesek.
- A `topological_sort()` visszaadja a task-ok helyes futási sorrendjét.

**Tipp:** ha saját DAG-ot írsz, mindig ellenőrizd `DagBag`-gel, mielőtt commitolod!
```python
assert not bag.import_errors, f'DAG import hibák: {bag.import_errors}'
```

## 5. `default_args` – minden task alapértelmezései

A `default_args` egy Python dict, amelyet a `DAG(...)` konstruktornak adunk át.
Az itt megadott értékek **minden task-ra érvényesek**, kivéve, ha a task felülírja.

```python
default_args = {
    'owner': 'data-team',        # Ki a felelős? (UI-on látszik)
    'retries': 3,                # Hány újrapróbálkozás hiba esetén?
    'retry_delay': timedelta(minutes=5),  # Mennyi várakozás retry-ok között?
    'email_on_failure': True,    # Küldj emailt, ha a task megbukik?
    'email': ['alert@ceg.hu'],   # Hova küldj?
    'start_date': datetime(2024, 1, 1),  # Mikor kezdődjön az első run?
}
```

**start_date csapda:** ha `catchup=True` (alapértelmezett!), az Airflow az összes
elmulasztott run-t pótolni fogja a `start_date`-től napjainkig. Éles rendszernél
mindig állíts `catchup=False`-t, ha nem akarod a backfill-t.

In [7]:
# default_args demonstráció – mi történik, ha egy task felülírja az értéket?
from datetime import timedelta, datetime

default_args = {
    'owner':            'data-team',
    'retries':          3,
    'retry_delay':      timedelta(minutes=5),
    'email_on_failure': True,
    'email':            ['alert@ceg.hu'],
    'start_date':       datetime(2024, 1, 1),
}

# Egy task felülírhatja a default_args-t
task_overrides = {
    'retries': 1,           # csak 1 újrapróbálkozás ennél a task-nál
    'retry_delay': timedelta(seconds=30),
}

# Tényleges task konfig: default_args + override
def effective_config(defaults, overrides):
    cfg = {**defaults, **overrides}
    return cfg

eff = effective_config(default_args, task_overrides)

print('default_args értékek:')
for k, v in default_args.items():
    overridden = k in task_overrides
    marker = '  ← FELÜLÍRVA' if overridden else ''
    print(f'  {k:20s} = {str(v):<30s}{marker}')

print('\nEz a task tényleges konfigja:')
for k, v in eff.items():
    print(f'  {k:20s} = {v}')

default_args értékek:
  owner                = data-team                     
  retries              = 3                               ← FELÜLÍRVA
  retry_delay          = 0:05:00                         ← FELÜLÍRVA
  email_on_failure     = True                          
  email                = ['alert@ceg.hu']              
  start_date           = 2024-01-01 00:00:00           

Ez a task tényleges konfigja:
  owner                = data-team
  retries              = 1
  retry_delay          = 0:00:30
  email_on_failure     = True
  email                = ['alert@ceg.hu']
  start_date           = 2024-01-01 00:00:00


## 6. Klasszikus `PythonOperator` – task létrehozása

A klasszikus API-ban minden task egy **Operator** példány:

```python
from airflow.operators.python import PythonOperator

def extract_fn(**context):
    # context['ti'] = TaskInstance – XCom push/pull itt érhető el
    records = fetch_from_db()
    context['ti'].xcom_push(key='records', value=records)

extract = PythonOperator(
    task_id='extract',
    python_callable=extract_fn,
    dag=dag,
)
```

**Legfontosabb paraméterek:**
| Paraméter          | Leírás                                               |
|:-------------------|:-----------------------------------------------------|
| `task_id`          | Egyedi azonosító a DAG-on belül                      |
| `python_callable`  | A meghívandó Python függvény                         |
| `op_kwargs`        | Extra kulcsszó-argumentumok a callable-nek           |
| `retries`          | Felülírja a default_args értékét                     |
| `dag`              | Melyik DAG-hoz tartozik                              |

In [8]:
# PythonOperator-szerű task demo – Airflow nélkül futtatható
# Megmutatja a task végrehajtás lényegét: callable + context

class FakeTaskInstance:
    """Egyszerűsített TaskInstance – csak XCom push/pull"""
    def __init__(self):
        self._store = {}
    def xcom_push(self, key, value):
        self._store[key] = value
        print(f'    [XCom push] {key} = {str(value)[:50]}')
    def xcom_pull(self, task_ids, key):
        val = self._store.get(key)
        print(f'    [XCom pull] {key} = {str(val)[:50]}')
        return val

ti = FakeTaskInstance()
context = {'ti': ti, 'execution_date': '2024-01-15'}

# 3 task függvény – pontosan ahogy az etl_pipeline.py-ban van
def extract_fn(**ctx):
    records = [{'id': i, 'amount': i * 100.0} for i in range(1, 6)]
    ctx['ti'].xcom_push('records', records)
    return len(records)

def transform_fn(**ctx):
    records = ctx['ti'].xcom_pull('extract', 'records')
    result = [{'id': r['id'], 'amount_huf': r['amount'] * 390} for r in records]
    ctx['ti'].xcom_push('transformed', result)

def load_fn(**ctx):
    data = ctx['ti'].xcom_pull('transform', 'transformed')
    print(f'    Betöltve: {len(data)} rekord → data warehouse')

# Szekvenciális futtatás (a >> operátor helyett)
print('Task-ok futtatása:')
for task_name, fn in [('extract', extract_fn), ('transform', transform_fn), ('load', load_fn)]:
    print(f'  {task_name}:')
    fn(**context)

Task-ok futtatása:
  extract:
    [XCom push] records = [{'id': 1, 'amount': 100.0}, {'id': 2, 'amount': 2
  transform:
    [XCom pull] records = [{'id': 1, 'amount': 100.0}, {'id': 2, 'amount': 2
    [XCom push] transformed = [{'id': 1, 'amount_huf': 39000.0}, {'id': 2, 'amou
  load:
    [XCom pull] transformed = [{'id': 1, 'amount_huf': 39000.0}, {'id': 2, 'amou
    Betöltve: 5 rekord → data warehouse


### A `>>` operátor – dependency chain

Az Airflow-ban a task-ok közötti sorrendet a `>>` (és `<<`) operátorral adjuk meg:

```python
extract >> transform >> load
# Ekvivalens:
extract.set_downstream(transform)
transform.set_downstream(load)
```

**Párhuzamos ágak:**
```python
transform >> [load_dw, load_mart]   # transform után MINDKÉT task elindul
[load_dw, load_mart] >> notify      # notify csak akkor indul, ha MINDKÉT kész
```

A `>>` operátor Pythonban az `__rshift__` magic method – az Airflow `BaseOperator` implementálja.

In [9]:
# Dependency chain szimulálása – >> operátor logikájának bemutatása
from collections import defaultdict

class Task:
    """Minimális task osztály a >> operátor demonstrálásához"""
    def __init__(self, task_id):
        self.task_id = task_id
        self.downstream = []
        self.upstream   = []

    def __rshift__(self, other):
        """self >> other: self előbb fut, mint other"""
        if isinstance(other, list):
            for t in other:
                self.downstream.append(t)
                t.upstream.append(self)
        else:
            self.downstream.append(other)
            other.upstream.append(self)
        return other  # láncolhatóság: a >> b >> c

    def __repr__(self):
        return self.task_id

# Pipeline felépítése – pontosan úgy, ahogy Airflow DAG-ban
extract   = Task('extract')
transform = Task('transform')
load_dw   = Task('load_dw')
load_mart = Task('load_mart')
notify    = Task('notify')

extract >> transform >> [load_dw, load_mart]
load_dw  >> notify
load_mart >> notify

# Dependency gráf kiírása
all_tasks = [extract, transform, load_dw, load_mart, notify]
print('Dependency gráf:')
for t in all_tasks:
    up  = [u.task_id for u in t.upstream]
    dn  = [d.task_id for d in t.downstream]
    print(f'  {t.task_id:12s}  upstream={up}  downstream={dn}')

Dependency gráf:
  extract       upstream=[]  downstream=['transform']
  transform     upstream=['extract']  downstream=['load_dw', 'load_mart']
  load_dw       upstream=['transform']  downstream=['notify']
  load_mart     upstream=['transform']  downstream=['notify']
  notify        upstream=['load_dw', 'load_mart']  downstream=[]


## 7. TaskFlow API – `@task` dekorátor

Az Airflow 2.0-ban bevezetett TaskFlow API leegyszerűsíti a DAG írást:

**Klasszikus API:**
```python
def extract_fn(**context):
    records = fetch_data()
    context['ti'].xcom_push('records', records)   # manuális XCom push

extract = PythonOperator(task_id='extract', python_callable=extract_fn, dag=dag)
```

**TaskFlow API:**
```python
@task
def extract() -> list:
    return fetch_data()   # return → automatikus XCom push

# Híváskor: records = extract()  → automatikus XCom pull
```

A `@task` dekorátor a háttérben ugyanúgy `PythonOperator`-t hoz létre,
de elrejti az XCom push/pull boilerplate-t.

In [10]:
# TaskFlow DAG betöltése és struktúrájának megmutatása
from airflow.models import DagBag

bag2 = DagBag(dag_folder='dags', include_examples=False)
dag  = bag2.dags.get('etl_taskflow')

if dag:
    print('etl_taskflow task-ok (topológiai sorrendben):')
    for task in dag.topological_sort():
        ups = [u.task_id for u in task.upstream_list]
        print(f'  {task.task_id:20s}  retries={task.retries}  upstream={ups}')
else:
    print('etl_taskflow nem található – ellenőrizd a dags/ mappát')

[2026-03-31T07:37:30.951+0000] {dagbag.py:545} INFO - Filling up the DagBag from dags
[2026-03-31T07:37:31.152+0000] {dagbag.py:514} ERROR - Exception bagging dag: etl_pipeline
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/site-packages/airflow/models/dagbag.py", line 505, in _bag_dag
    raise AirflowDagDuplicatedIdException(
airflow.exceptions.AirflowDagDuplicatedIdException: Ignoring DAG etl_pipeline from dags/.ipynb_checkpoints/etl_pipeline-checkpoint.py - also found in dags/etl_pipeline.py
[2026-03-31T07:37:31.153+0000] {dagbag.py:452} ERROR - Failed to bag_dag: dags/.ipynb_checkpoints/etl_pipeline-checkpoint.py
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/site-packages/airflow/models/dagbag.py", line 448, in _process_modules
    self.bag_dag(dag=dag, root_dag=dag)
  File "/opt/conda/lib/python3.11/site-packages/airflow/models/dagbag.py", line 467, in bag_dag
    self._bag_dag(dag=dag, root_dag=root_dag, recursive=True)
  File "/opt/c

### Mi változott? Klasszikus API vs TaskFlow API

| Szempont              | Klasszikus (`PythonOperator`) | TaskFlow (`@task`)            |
|:----------------------|:------------------------------|:------------------------------|
| XCom push             | `ti.xcom_push(key, value)`    | `return value`                |
| XCom pull             | `ti.xcom_pull(task_ids, key)` | automatikus (paraméterként)   |
| Task definíció        | `PythonOperator(task_id=…)`   | `@task` dekorátor             |
| Dependency            | `t1 >> t2`                    | `t2(t1())` – Python hívás     |
| Kód mennyisége        | több boilerplate               | tömörebb, olvashatóbb         |
| Visszafelé kompatibil | igen                          | Airflow 2.0+                  |
| Dinamikus task-ok     | nehezkes (`PythonOperator`)   | `@task.expand()` – egyszerű   |

**Mikor maradj a klasszikus API-nál?** Ha más Operator típust (pl. `BashOperator`,
`SparkSubmitOperator`, `S3ToRedshiftOperator`) használsz – azok nem `@task`-kal működnek.

## 8. XCom – task-ok közötti adatátadás

Az **XCom** (Cross-Communication) az a mechanizmus, amellyel egy task kisméretű adatot
ad át a következőnek. Az XCom értékeket az Airflow a **Metadata DB-ben** tárolja (pickle vagy JSON).

**Hogyan működik:**
1. Task A: `ti.xcom_push(key='result', value=42)` → Metadata DB INSERT
2. Task B: `ti.xcom_pull(task_ids='task_a', key='result')` → Metadata DB SELECT

**Fontos korlát:** az XCom adatot a DB-ben tárolja → **ne tárolj benne nagy adatot!**
Az SQLite alapértelmezésben 2 GB-os BLOB-ot enged, de a PostgreSQL is lassul néhány MB felett.

In [11]:
# XCom viselkedés demonstrálása standalone módban
# (Airflow nélkül – a Metadata DB-t dict-tel szimulálva)
import random, json

class XComStore:
    """Egyszerűsített XCom szimuláció – pontosan a DB séma logikáját követi"""
    def __init__(self):
        # Kulcs: (dag_id, run_id, task_id, key)
        self._db = {}

    def push(self, task_id, key, value):
        serialized = json.dumps(value)  # Airflow JSON-ként szerializál
        size_kb = len(serialized.encode()) / 1024
        self._db[(task_id, key)] = value
        print(f'  XCom push [{task_id}][{key}]: {size_kb:.1f} KB')

    def pull(self, task_id, key):
        val = self._db.get((task_id, key))
        print(f'  XCom pull [{task_id}][{key}] → {str(val)[:60]}')
        return val

xcom = XComStore()

# Extract task – push-olja a rekordokat
records = [{'id': i, 'amount': round(random.uniform(100, 50000), 2)} for i in range(1, 11)]
xcom.push('extract', 'records', records)
xcom.push('extract', 'record_count', len(records))

# Transform task – pull-olja és feldolgozza
pulled = xcom.pull('extract', 'records')
count  = xcom.pull('extract', 'record_count')
print(f'\nFeldolgozva: {count} rekord')
print('Első 3:', pulled[:3])

  XCom push [extract][records]: 0.3 KB
  XCom push [extract][record_count]: 0.0 KB
  XCom pull [extract][records] → [{'id': 1, 'amount': 41207.58}, {'id': 2, 'amount': 7309.08}
  XCom pull [extract][record_count] → 10

Feldolgozva: 10 rekord
Első 3: [{'id': 1, 'amount': 41207.58}, {'id': 2, 'amount': 7309.08}, {'id': 3, 'amount': 2705.79}]


### XCom korlátai – mikor NE használjuk?

Az XCom a Metadata DB-ben tárol → **nagy adatot soha ne adj át XCom-on!**

| Adatméret       | Ajánlott megközelítés                           |
|:----------------|:------------------------------------------------|
| < 1 KB          | XCom – tökéletes (pl. szám, státusz, path)      |
| 1 KB – 1 MB     | XCom – elfogadható, de figyelj rá               |
| > 1 MB          | **NE használj XCom-ot!** Adj át fájl elérési utat |
| DataFrame, tömb | Mentsd Parquet-be, XCom-ban csak a path-t tárold |

**Bevett gyakorlat:**
```python
@task
def extract() -> str:
    df = fetch_large_dataframe()
    path = '/tmp/orders_2024-01-15.parquet'
    df.to_parquet(path)         # ← adat a fájlrendszeren
    return path                 # ← XCom csak a path-t kapja

@task
def transform(path: str) -> str:
    df = pd.read_parquet(path)  # ← fájlból olvas, nem XCom-ból
    ...
```

In [12]:
# XCom méret demonstráció – nagy adat vs path átadás
import json, sys

# Szimuláljuk: 10 000 rekord XCom-on keresztül
large_records = [{'id': i, 'name': f'user_{i}', 'value': i * 3.14} for i in range(10_000)]
serialized     = json.dumps(large_records).encode('utf-8')
size_mb        = len(serialized) / 1024 / 1024

print(f'10 000 rekord JSON mérete: {size_mb:.2f} MB')
print(f'  → XCom-ban tárolva: minden run a DB-t terheli!')
print()

# Helyes megközelítés: csak a path-t tárold
fake_path = '/tmp/orders_2024-01-15.parquet'
path_size = sys.getsizeof(fake_path)
print(f'Path string mérete: {path_size} bájt')
print(f'  → 10 000-szeres különbség!')
print()
print('Ökölszabály: XCom-on csak skalár / rövid string / path menjen.')
print('Nagy adatot: S3 / GCS / helyi fájlrendszer / DuckDB-ben tárold.')

10 000 rekord JSON mérete: 0.53 MB
  → XCom-ban tárolva: minden run a DB-t terheli!

Path string mérete: 79 bájt
  → 10 000-szeres különbség!

Ökölszabály: XCom-on csak skalár / rövid string / path menjen.
Nagy adatot: S3 / GCS / helyi fájlrendszer / DuckDB-ben tárold.


## 9. Retry és hibakezelés

Az Airflow automatikusan kezeli a hibás task-okat:

```
Task fut
  ↓ kivétel
up_for_retry  (sárga) → vár retry_delay-t
  ↓ újraindul
Task fut ismét
  ↓ ha sikerül → success (zöld)
  ↓ ha megbukik és retry kimerült → failed (piros)
```

**Az `etl_taskflow.py` transform task-ja 10% eséllyel `RuntimeError`-t dob** –
szimulálva egy flaky külső API-t vagy ideiglenesen elérhetetlen adatforrást.

**Airflow UI-n látható állapotok:**
1. Task sárga: `up_for_retry`
2. Vár `retry_delay`-t (alapból 5 perc, demo: 10 s)
3. Újraindul: ha sikerül → zöld; ha megbukik → ismét sárga
4. 3 retry után → piros: `failed`

In [13]:
# Retry logika szimulálása – Airflow nélkül
import time, random

def flaky_transform(records, max_retries=3, fail_prob=0.50):
    """Szimulálja az Airflow retry mechanizmusát.
    fail_prob: annak valószínűsége, hogy egy kísérlet megbukik (50% demo módban)
    """
    for attempt in range(max_retries + 1):
        try:
            if random.random() < fail_prob:
                raise RuntimeError('Szimulált flaky hiba (pl. timeout)')
            # Sikeres transform: forint átváltás
            result = [{'id': r['id'], 'amount_huf': r['amount'] * 390} for r in records]
            print(f'  Sikeres a {attempt + 1}. próbán – {len(result)} rekord feldolgozva')
            return result
        except RuntimeError as e:
            if attempt < max_retries:
                delay = 0.1 * (2 ** attempt)  # exponential backoff
                print(f'  {attempt + 1}. kísérlet HIBA: {e}')
                print(f'    → retry_delay: {delay:.1f}s (Airflow-ban percek!)')
                time.sleep(delay)
            else:
                print(f'  Minden retry ({max_retries}x) kimerült → task FAILED')
                raise

random.seed(42)
records = [{'id': i, 'amount': 1000.0} for i in range(5)]
try:
    flaky_transform(records)
except RuntimeError:
    print('  (Hiba elkapva – az Airflow piros task-ot mutatna)')

  Sikeres a 1. próbán – 5 rekord feldolgozva


### `on_failure_callback` – mikor hívódik meg?

A `on_failure_callback` akkor fut le, amikor a task **véglegesen megbukik**
(az összes retry kimerült). Tipikus felhasználás: Slack értesítés, PagerDuty alert, DB naplózás.

```python
def notify_slack(context):
    dag_id  = context['dag'].dag_id
    task_id = context['task'].task_id
    run_id  = context['run_id']
    # slack_sdk.WebClient(...).chat_postMessage(...)
    print(f'Slack: {dag_id}.{task_id} FAILED (run_id={run_id})')

transform = PythonOperator(
    task_id='transform',
    python_callable=transform_fn,
    on_failure_callback=notify_slack,   # ← csak végső hiba esetén
    dag=dag,
)
```

In [14]:
# on_failure_callback szimuláció – mi kerül a context-be?
import random, time
from datetime import datetime

def on_failure_callback(context):
    """Airflow ezt hívja meg, ha a task véglegesen megbukik."""
    print('=== on_failure_callback meghívva ===')
    print(f'  DAG:           {context["dag_id"]}')
    print(f'  Task:          {context["task_id"]}')
    print(f'  Run ID:        {context["run_id"]}')
    print(f'  Hiba:          {context["exception"]}')
    print(f'  Időbélyeg:     {context["ts"]}')
    print('  → Slack/PagerDuty/email értesítés küldése...')

def run_with_callback(task_fn, context, max_retries=2):
    """Airflow Scheduler viselkedés szimulálása: retry + callback"""
    for attempt in range(max_retries + 1):
        try:
            task_fn()
            print(f'Task sikeres ({attempt + 1}. próba)')
            return
        except Exception as e:
            context['exception'] = e
            if attempt < max_retries:
                print(f'  Retry {attempt + 1}/{max_retries}...')
            else:
                on_failure_callback(context)  # ← csak itt hívódik!

ctx = {
    'dag_id':   'etl_taskflow',
    'task_id':  'transform',
    'run_id':   'scheduled__2024-01-15T06:00:00+00:00',
    'ts':       datetime.now().isoformat(),
    'exception': None,
}

random.seed(99)  # Garantáltan minden retry megbukik
run_with_callback(lambda: (_ for _ in ()).throw(RuntimeError('DB timeout')), ctx)

  Retry 1/2...
  Retry 2/2...
=== on_failure_callback meghívva ===
  DAG:           etl_taskflow
  Task:          transform
  Run ID:        scheduled__2024-01-15T06:00:00+00:00
  Hiba:          DB timeout
  Időbélyeg:     2026-03-31T07:37:52.194734
  → Slack/PagerDuty/email értesítés küldése...


## 10. `execution_date` vs valós futási idő

Ez az egyik legzavaróbb fogalom Airflow-ban:

| Fogalom            | Mit jelent                                                      |
|:-------------------|:----------------------------------------------------------------|
| `execution_date`   | Az **időszak kezdete**, amelyre a DAG vonatkozik                |
| `data_interval_end`| Az időszak vége (= execution_date + schedule_interval)          |
| Tényleges futás    | A `data_interval_end` **után** indul el                         |

**Példa:** napi DAG, `schedule='@daily'`, `start_date=2024-01-01`
- `execution_date=2024-01-01` → a 2024-01-01 nap adatát dolgozza fel
- Tényleges futás: **2024-01-02 00:00** (amikor a nap már lezárult!)

**Következmény:** az Airflow mindig a *múlt* adatát dolgozza fel egy futásban.
Ha `execution_date`-t használsz az adatszűrőben, ez szándékos és helyes.

In [17]:
# execution_date vs tényleges futási idő – vizuális táblázat
from datetime import datetime, timedelta

schedule_interval = timedelta(days=1)   # @daily
start_date        = datetime(2024, 1, 1)

print(f'  {"execution_date":22}  {"data_interval_end":22}  {"tényleges futás kb.":22}')
print('-' * 80)

for run_num in range(1, 6):
    execution_date   = start_date + schedule_interval * (run_num - 1)
    interval_end     = execution_date + schedule_interval
    actual_start     = interval_end  # Airflow az interval_end után indítja
    print(
        f'{run_num:>3}  '
        f'{str(execution_date.date()):22}  '
        f'{str(interval_end.date()):22}  '
        f'{str(actual_start.date()):22}'
    )

print()
print('Magyarázat: az 1. run a 2024-01-01 nap adatát dolgozza fel,')
print('de csak 2024-01-02-án indul el (amikor a nap már lezárult).')

  execution_date          data_interval_end       tényleges futás kb.   
--------------------------------------------------------------------------------
  1  2024-01-01              2024-01-02              2024-01-02            
  2  2024-01-02              2024-01-03              2024-01-03            
  3  2024-01-03              2024-01-04              2024-01-04            
  4  2024-01-04              2024-01-05              2024-01-05            
  5  2024-01-05              2024-01-06              2024-01-06            

Magyarázat: az 1. run a 2024-01-01 nap adatát dolgozza fel,
de csak 2024-01-02-án indul el (amikor a nap már lezárult).


## 11. Airflow UI ellenőrzés

1. Nyisd meg: **http://localhost:8080** (admin / admin)
2. DAGs listán keresd: `etl_pipeline` és `etl_taskflow`
3. Kapcsold be a DAG-ot (toggle bal oldalt) – Airflow alapban kikapcsolt állapotban tölt be
4. Kattints **Trigger DAG** → Graph View-ban kövesd a futást
5. Kattints egy task-ra → **Log** fülön lásd a print kimeneteket és az XCom értékeket

**Hasznos nézetek:**
| Nézet       | Mire jó                                                          |
|:------------|:-----------------------------------------------------------------|
| Graph View  | Task-ok és függőségek vizuálisan; aktuális állapotok             |
| Grid View   | Historikus futások – melyik run sikerült / bukott meg            |
| Gantt Chart | Párhuzamos task-ok időbeli lefutása – bottleneck keresés         |
| XCom        | Admin → XCom menü – az összes tárolt XCom érték megtekinthető   |

## 12. Összefoglalás – mikor NEM érdemes Airflow-t használni?

Az Airflow erős eszköz, de nem minden esetben a legjobb választás:

**Airflow IGEN:**
- Komplex, sok-lépéses ETL/ELT pipeline-ok
- Számos függőséggel rendelkező workflow-ok
- Szükség van UI-ra, naplózásra, riasztásokra
- Backfill és catchup szükséges (historikus adat újrafeldolgozás)
- Nagy csapat, sok DAG, governance szükséges

**Airflow NEM:**
- Egyszerű, 1-2 lépéses job → cron + bash script is elég
- Valós idejű / streaming feldolgozás → Kafka + Flink / Spark Streaming
- Microsecond latency → Airflow overhead túl nagy
- Kis csapat, kevés pipeline → Prefect / Dagster kisebb overhead-del
- Adat-orientált lineage fontos → dbt (aminek van beépített DAG-ja)

**Ökölszabály:** ha a pipeline 3 lépésnél bonyolultabb, van retry-igény,
vagy mások is figyelik a futásokat → érdemes orchestratort használni.